# ABCA superconductivity at one displacement and density

Free eight-band structure → constrained RPA → all-transfer particle-hole parquet → odd intravalley pairing.

Install the package using the README first, and select that Python environment as this notebook's kernel. **Run all cells in order.** Results are written under `results/` relative to this notebook's working directory. Change `DELTA_D_MEV` and `DENSITY_1E11_CM2` below for a different physical point. No precomputed vertex is required.

The full M=10 calculation needs roughly 12 GB for solver arrays plus process overhead; use a machine with additional free memory. `SMOKE_TEST=True` selects a small, thermally broadened installation check, not a converged physical approximation.

In [ ]:
import os
for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(name, "1")
from pathlib import Path
import gc, json, sys
import matplotlib.pyplot as plt
from IPython.display import display
from threadpoolctl import threadpool_limits
from parquet_superconductivity.config import point_config, smoke_config, preview, result_directory
from parquet_superconductivity.workflow import stage1_screening, stage2_vertex, stage3_pairing
from parquet_superconductivity.plotting import plot_screening, plot_gap
from parquet_superconductivity.fixed_vertex_temperature import solve_fixed_vertex_temperature, plot_order_parameter
from parquet_superconductivity.reporting import save_pairing_figures
blas_limits = threadpool_limits(limits=1)
print("Python:", sys.executable)

## Parameters
`DELTA_D_MEV` is the outer-layer energy difference in meV. `DENSITY_1E11_CM2` is the density of the selected spin–valley flavor, in units of $10^{11}$ cm$^{-2}$. Temperatures are supplied as $k_B T$ in meV. The low-temperature solve holds the vertex fixed; it does not redo screening or parquet at that temperature.

In [ ]:
DELTA_D_MEV = 110.0
DENSITY_1E11_CM2 = 6.375
MESH_RADIUS = 10
TEMPERATURE_MEV = 0.1
GATE_NM = 25.0
DIELECTRIC = 15.0
PARQUET_TOLERANCE = 5e-3  # Acceptance threshold; inspect the actual residual.
SMOKE_TEST = False
RUN_LOW_TEMPERATURE = True
PAIRING_TEMPERATURE_MEV = 0.01
RESULTS_ROOT = Path("results")

config = point_config(DELTA_D_MEV, DENSITY_1E11_CM2, mesh_radius=MESH_RADIUS)
config["model"].update(temperature_eV=TEMPERATURE_MEV/1000,
                       gate_distance_nm=GATE_NM, dielectric_constant=DIELECTRIC)
config["solver"]["tolerance"] = PARQUET_TOLERANCE
if SMOKE_TEST:
    config = smoke_config()
    RUN_LOW_TEMPERATURE = False
    print("SMOKE TEST: M=2, kBT=5 meV; installation check only.")
OUTPUT = result_directory(config, base=RESULTS_ROOT).resolve()
print("Output:", OUTPUT)
print(json.dumps(preview(config), indent=2))

## 1. Constrained screening
The active–active transition is excluded from the eight-band polarization. The other 63 ordered transitions, including mixed active/remote terms, screen both the direct and exchange source interactions.

In [ ]:
system, crpa, _ = stage1_screening(config, OUTPUT, progress=True)
fig = plot_screening(crpa)
fig.savefig(OUTPUT / "01_screening" / "screening.pdf")
plt.show()

## 2. Coupled particle-hole parquet equations
This is the expensive stage. Rerunning resumes the last converged checkpoint **in this same output directory**. The full vertex is reconstructed only after reaching the target coupling. If continuation stops early, inspect its residuals; the pairing stage must not use a failed iterate. Run only one process per output directory.

In [ ]:
_, problem, vertex, gamma, _ = stage2_vertex(config, OUTPUT, progress=True, resume=True)
print(json.dumps({key: gamma.metadata[key] for key in (
    "interaction_scale", "gamma_ph_relative_residual", "bse_relative_residual",
    "crossing_defect", "hermiticity_defect")}, indent=2))
gamma.flat._mmap.close()
del problem, vertex, gamma
gc.collect()

## 3. Cooper eigenvalue and nonlinear gap
The linearized eigenmode is normalized arbitrarily; its magnitude is not an energy gap. A nonlinear order parameter is solved when the leading Cooper eigenvalue exceeds the instability threshold. Its gap and filling residuals must converge. The real/imaginary plots use the solver's band gauge with one constant phase rotation; the phase-winding plot separately fixes the $A_1$-positive band gauge.

In [ ]:
pair, sc, _ = stage3_pairing(config, OUTPUT, progress=True)
print("Cooper lambda_max:", sc.linearized.leading_eigenvalue)
print("Classification:", sc.linearized.classification)
print("Eigen residual:", sc.linearized.residual)
if not sc.linearized.converged:
    raise RuntimeError("Linearized pairing did not converge.")
if sc.nonlinear is not None:
    print("Nonlinear status:", sc.nonlinear.status)
    print("Max gap (meV):", 1000*sc.nonlinear.max_gap)
    print("Gap / filling residual:", sc.nonlinear.gap_residual, sc.nonlinear.filling_residual)
    if not sc.nonlinear.converged:
        raise RuntimeError("Nonlinear gap did not converge; inspect saved diagnostics.")
save_pairing_figures(system, sc, OUTPUT / "03_superconductivity")
display(plot_gap(system, sc))
if sc.nonlinear is not None and sc.nonlinear.converged:
    display(plot_order_parameter(system, sc))
plt.close("all")
del pair
gc.collect()

## 4. Optional lower-temperature gap and phase winding
The original active-band population is held fixed, and the chemical potential is readjusted. A winding is reported only for a nonzero converged gap on a resolved closed normal-state Fermi contour. The winding in a specified band gauge is not a BdG Chern number.

In [ ]:
if RUN_LOW_TEMPERATURE:
    low_system, low_sc, low_summary, low_output = solve_fixed_vertex_temperature(
        OUTPUT, PAIRING_TEMPERATURE_MEV/1000,
        nonlinear_kwargs=config["pairing"]["nonlinear_kwargs"], progress=True)
    print(json.dumps({key: low_summary[key] for key in (
        "lambda_max", "status", "max_gap_meV", "gap_residual", "filling_residual")}, indent=2))
    save_pairing_figures(low_system, low_sc, low_output)
    display(plot_order_parameter(low_system, low_sc))
    if low_sc.nonlinear.is_superconducting:
        from parquet_superconductivity.phase_winding import plot_saved_phase_winding
        if (low_output / "phase_winding.json").exists():
            fig, report = plot_saved_phase_winding(low_output / "superconductivity.npz")
            display(fig)
            print(json.dumps(report["loops"], indent=2))
    plt.close("all")
else:
    print("Optional lower-temperature solve disabled.")

## Saved outputs
Each run includes its complete configuration, source hashes, screening data, continuation/checkpoint history, explicit full vertex, Cooper matrix, eigenmode, nonlinear gap, number-equation residual, and PNG/PDF figures. Large generated arrays are excluded from Git by `.gitignore`. See `docs/METHODS.md` for momentum and normalization conventions.